In [376]:
import pandas as pd
from difflib import get_close_matches
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pickle
import ast

In [377]:
# Load the data

movie_metadata = pd.read_csv('data/movies_metadata.csv', low_memory=False)
links = pd.read_csv('data/links.csv')
ratings = pd.read_csv('data/ratings.csv')
keywords = pd.read_csv('data/keywords.csv')
credits = pd.read_csv('data/credits.csv')

In [378]:
# Correct merge: ratings (MovieLens movieId) -> links (TMDB tmdbId) -> movies_metadata (TMDB id)

movie_metadata['id'] = pd.to_numeric(movie_metadata['id'], errors='coerce') # change to numeric, coerce errors to NaN
movie_metadata = movie_metadata.dropna(subset=['id']).copy() # drop rows where 'id' is NaN
movie_metadata['id'] = movie_metadata['id'].astype(int) # change 'id' column to int type

links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce') 
links = links.dropna(subset=['tmdbId']).copy() 
links['tmdbId'] = links['tmdbId'].astype(int)

# Merge ratings with links on MovieLens movieId to get TMDB tmdbId
movie_data = ratings.merge(links[['movieId', 'tmdbId']], on='movieId', how='inner')
# Merge with metadata on TMDB id to get titles and metadata
movie_data = movie_data.merge(movie_metadata[['id', 'title']], left_on='tmdbId', right_on='id', how='inner')

# Keep only necessary columns for recommendations
movie_ratings = movie_data[['userId', 'movieId', 'rating', 'title']]

In [379]:
# Extract Director from crew & remove spaces from cast names for content-based filtering
def extract_director(crew_text):
    if pd.isna(crew_text) or crew_text == '':
        return ''
    
    try:
        crew_list = ast.literal_eval(crew_text)
        for member in crew_list:
            if member.get('job') == 'Director':
                return member.get('name', '')
        return ''
    except:
        return ''

def remove_space(text):
    if pd.isna(text):
        return ''
    return text.replace(' ', '')

credits['director'] = credits['crew'].apply(extract_director)
credits['director'] = credits['director'].apply(remove_space)
print(credits.head())

                                                cast  \
0  [{'cast_id': 14, 'character': 'Woody (voice)',...   
1  [{'cast_id': 1, 'character': 'Alan Parrish', '...   
2  [{'cast_id': 2, 'character': 'Max Goldman', 'c...   
3  [{'cast_id': 1, 'character': "Savannah 'Vannah...   
4  [{'cast_id': 1, 'character': 'George Banks', '...   

                                                crew     id        director  
0  [{'credit_id': '52fe4284c3a36847f8024f49', 'de...    862    JohnLasseter  
1  [{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...   8844     JoeJohnston  
2  [{'credit_id': '52fe466a9251416c75077a89', 'de...  15602    HowardDeutch  
3  [{'credit_id': '52fe44779251416c91011acb', 'de...  31357  ForestWhitaker  
4  [{'credit_id': '52fe44959251416c75039ed7', 'de...  11862    CharlesShyer  


In [380]:
#Fix genre, keyword, cast, and crew formatting (json to list) for content based filtering

def extract_names(text):
    if pd.isna(text) or text == '':
        return ''

    try:
        items = ast.literal_eval(text)
        return ', '.join(item['name'] for item in items if 'name' in item)
    except:
        return ''

def extract_top_cast(cast_text, n=5):
    if pd.isna(cast_text) or cast_text == '':
        return ''

    try:
        cast_list = ast.literal_eval(cast_text)

        # Ensure sorted by billing order
        cast_list = sorted(cast_list, key=lambda x: x.get('order', 999))

        return ' '.join(
            actor['name'].replace(' ', '')
            for actor in cast_list[:n]
        )
    except:
        return ''

# Genres
movie_metadata['genres'] = movie_metadata['genres'].apply(extract_names)

# Keywords
keywords['keywords'] = keywords['keywords'].apply(extract_names)

#Cast & Crew
credits['cast'] = credits['cast'].apply(extract_top_cast)
credits['crew'] = credits['crew'].apply(extract_names)

In [381]:
# Keyword data preprocessing (for content-based filtering)
keyword_data = keywords.copy()
keyword_data['id'] = pd.to_numeric(keyword_data['id'], errors='coerce') 
keyword_data = keyword_data.dropna(subset=['id']).copy() # drop rows where 'id' is NaN
keyword_data['id'] = keyword_data['id'].astype(int) 
keyword_data = ( 
    keyword_data
    .merge( # merge keywords with links to get MovieLens movieId
        links[['movieId', 'tmdbId']], 
        left_on='id', 
        right_on='tmdbId', 
        how='inner')
    .merge( # merge with movie metadata to get titles and genres
        movie_metadata[['id', 'title', 'genres']], 
        on='id', 
        how='inner')
    [['tmdbId', 'title', 'genres', 'keywords']]
    .dropna(subset=['title'])
    .drop_duplicates(subset=['tmdbId'])
    .reset_index(drop=True)
 ) 

keyword_data['content'] = ( # combine genres and keywords into a single string for content-based filtering
    keyword_data['genres'].fillna('') + ' ' +
    keyword_data['keywords'].fillna('')
)
print(keyword_data.head())

   tmdbId                        title                      genres  \
0     862                    Toy Story   Animation, Comedy, Family   
1    8844                      Jumanji  Adventure, Fantasy, Family   
2   15602             Grumpier Old Men             Romance, Comedy   
3   31357            Waiting to Exhale      Comedy, Drama, Romance   
4   11862  Father of the Bride Part II                      Comedy   

                                            keywords  \
0  jealousy, toy, boy, friendship, friends, rival...   
1  board game, disappearance, based on children's...   
2  fishing, best friend, duringcreditsstinger, ol...   
3  based on novel, interracial relationship, sing...   
4  baby, midlife crisis, confidence, aging, daugh...   

                                             content  
0  Animation, Comedy, Family jealousy, toy, boy, ...  
1  Adventure, Fantasy, Family board game, disappe...  
2  Romance, Comedy fishing, best friend, duringcr...  
3  Comedy, Drama, Roma

In [382]:
# Cast Data Preprocessing (for content-based filtering)
content_data = keyword_data.merge(credits[['id', 'cast', 'director']], left_on='tmdbId', right_on='id', how='left')

for col in ['genres', 'keywords', 'cast', 'director']: # remove commas from genres, keywords, cast, and director columns to avoid issues with string matching
    content_data[col] = (
        content_data[col]
        .fillna('')
        .str.replace(',', '', regex=False)
    )

content_data['content'] = ( # combine genres, keywords, cast, and crew into a single string for content-based filtering
    
    # Repeat columns multiple times to give them weights. 
    content_data['genres'].fillna('') + ' ' + # Genres: x3
    content_data['genres'].fillna('') + ' ' + 
    content_data['genres'].fillna('') + ' ' + 


    content_data['keywords'].fillna('') + ' ' + # Keywords: x3
    content_data['keywords'].fillna('') + ' ' +
    content_data['keywords'].fillna('') + ' ' +

    content_data['cast'].fillna('') + ' ' + # Cast: x2
    content_data['cast'].fillna('') + ' ' +

    content_data['director'].fillna('')# Director: x1
)

print(content_data['content'][0])

Animation Comedy Family Animation Comedy Family Animation Comedy Family jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life TomHanks TimAllen DonRickles JimVarney WallaceShawn TomHanks TimAllen DonRickles JimVarney WallaceShawn JohnLasseter


In [383]:
# Collaborative Filtering Approach

# Create a sparse matrix for user-item interactions

user_codes = movie_ratings['userId'].astype('category').cat.codes 
movie_codes = movie_ratings['movieId'].astype('category').cat.codes

user_item_sparse = csr_matrix(
    (movie_ratings['rating'], (user_codes, movie_codes))
)

movie_user_sparse = user_item_sparse.T

In [384]:
movie_index = movie_ratings['movieId'].astype('category').cat.categories # get unique movieIds in the order they appear in the sparse matrix

movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_index)} # create a mapping from movieId to index in the sparse matrix
idx_to_movie = {idx: movie_id for movie_id, idx in movie_to_idx.items()} # create a mapping from index in the sparse matrix to movieId

movie_lookup = ( # create a lookup table for movieId to title
    movie_ratings[['movieId', 'title']]
    .dropna(subset=['title'])
    .drop_duplicates(subset=['movieId'])
    .set_index('movieId')
)

In [385]:
# used for hybrid recommendation system to map movieId to tmdbId

movie_id_map = (
    links[['movieId', 'tmdbId']]
    .dropna(subset=['movieId', 'tmdbId'])
    .drop_duplicates(subset=['movieId'])
)

In [386]:
# Save the processed data to a pickle file for flask app usage

with open('recommendation_data.pkl', 'wb') as f: 
    pickle.dump({
        'movie_user_sparse': movie_user_sparse,
        'movie_index': movie_index,
        'movie_lookup': movie_lookup,
        'movie_to_idx': movie_to_idx,
        'idx_to_movie': idx_to_movie,
        'content_data': content_data,
        'movie_id_map': movie_id_map
    }, f)

In [387]:
#KNN Approach (Collaborative Filtering)
knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10, n_jobs=-1)
knn.fit(movie_user_sparse)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [388]:
# SVD Approach (Collaborative Filtering)
svd_components = 100
svd = TruncatedSVD(n_components=svd_components, random_state=42)
movie_factors = svd.fit_transform(movie_user_sparse)

with open('svd_model.pkl', 'wb') as file:
    pickle.dump(svd, file)

In [389]:
#Find optimal number of SVD components by checking explained variance for different k values, optimal value is 100

"""k_values = [10, 20, 50, 100, 150, 200]

results = []

for k in k_values:
    svd = TruncatedSVD(n_components=k, random_state=42)
    movie_factors = svd.fit_transform(movie_user_sparse)

    explained = svd.explained_variance_ratio_.sum()

    results.append((k, explained))

print(results)"""


'k_values = [10, 20, 50, 100, 150, 200]\n\nresults = []\n\nfor k in k_values:\n    svd = TruncatedSVD(n_components=k, random_state=42)\n    movie_factors = svd.fit_transform(movie_user_sparse)\n\n    explained = svd.explained_variance_ratio_.sum()\n\n    results.append((k, explained))\n\nprint(results)'

In [390]:
# find movieId by title with exact and fuzzy matching

def find_movie_id(movie_name, lookup_df):
    """Find the movieId for a given movie title, using exact and fuzzy matching. Returns None if no match is found."""
    titles = lookup_df['title']

    exact_match = titles[titles.str.lower() == movie_name.lower()]
    if not exact_match.empty:
        return exact_match.index[0]

    close = get_close_matches(movie_name, titles.tolist(), n=1, cutoff=0.5)
    if close:
        matched_title = close[0]
        return titles[titles == matched_title].index[0]

    return None

In [391]:
# Content-Based Filtering Approach
vectorizer = TfidfVectorizer(
    stop_words='english',
    min_df=2,
    max_df=0.7,
    ngram_range=(1, 2)
    )
content_matrix = vectorizer.fit_transform(content_data['content'].fillna(''))
content_movie_ids = content_data['tmdbId']
content_lookup = content_data.set_index('tmdbId')

def get_content_based_recommendations(movie_name, movie_features, lookup_df, n_recs=10):
    """Return a dataframe of top-N similar movies for a given movie title using content-based filtering."""
    
    movie_id = find_movie_id(movie_name, lookup_df)
    if movie_id is None:
        raise ValueError(f"Movie '{movie_name}' not found.")
    
    movie_positions = content_movie_ids[content_movie_ids == movie_id].index
    if len(movie_positions) == 0:
        raise ValueError(f"Movie '{movie_name}' not found in keyword features.")
    movie_idx = movie_positions[0]
    
    # Get the feature vector for the selected movie
    movie_features_vector = movie_features[movie_idx]
    
    # Calculate cosine similarity between the selected movie and all other movies
    similarities = cosine_similarity(movie_features_vector, movie_features).flatten()
    
    # Get the indices of the most similar movies
    similar_indices = similarities.argsort()[::-1][1:n_recs+1]  # Exclude the movie itself
    
    # Create a list of recommended movies
    recs = []
    for idx in similar_indices:
        rec_movie_id = int(content_movie_ids.iloc[idx])
        recs.append({
            'tmdbId': rec_movie_id,
            'Title': content_lookup.loc[rec_movie_id, 'title'],
            'Similarity': similarities[idx]
        })
    
    #print(f"Content-Based Recommendations for '{movie_name}' at movieId {movie_id}:")
    return pd.DataFrame(recs)

In [392]:
sample_title1 = 'Batman'
sample_title2 = 'Toy Story'
sample_title3 = 'Titanic'

content1 = get_content_based_recommendations(sample_title1, content_matrix, content_lookup, n_recs=5)
print(content1)
content2 = get_content_based_recommendations(sample_title2, content_matrix, content_lookup, n_recs=5)
print(content2)
content3 = get_content_based_recommendations(sample_title3, content_matrix, content_lookup, n_recs=5)
print(content3)

   tmdbId                                              Title  Similarity
0     415                                     Batman & Robin    0.784653
1     364                                     Batman Returns    0.437323
2  323027                  Justice League: Gods and Monsters    0.325105
3  411736              Batman: Return of the Caped Crusaders    0.302125
4  396330  LEGO DC Comics Super Heroes: Justice League - ...    0.288782
   tmdbId                       Title  Similarity
0  256835  Toy Story That Time Forgot    0.380240
1   10193                 Toy Story 3    0.361376
2   11551              Small Soldiers    0.334822
3   11359  The Indian in the Cupboard    0.306992
4   10585                Child's Play    0.287044
   tmdbId                Title  Similarity
0    8447   Angels and Insects    0.322455
1    2699              Titanic    0.318256
2  289523  Midnight Masquerade    0.268215
3  201283   Aaron Loves Angela    0.261007
4    9062           Love Story    0.254122


In [397]:
with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(content_matrix, f)

In [393]:
# Collaborative Filtering Recommendations

def get_collaborative_recommendations(movie_name, matrix, cf_model, lookup_df, model_type, n_recs=10):
    """Return a dataframe of top-N similar movies for a given movie title using collaborative filtering."""
    
    movie_id = find_movie_id(movie_name, lookup_df)
    if movie_id is None:
        raise ValueError(f"Movie '{movie_name}' not found.")
    movie_idx = movie_to_idx[movie_id]

    if model_type.lower() == 'knn':
        #print("Using KNN model for recommendations.")

        n_neighbors = min(n_recs + 1, matrix.shape[0]) # +1 because the closest neighbor is the movie itself
        distances, indices = cf_model.kneighbors( # KNN search
            matrix[movie_idx], # matrix[movie_idx] is the vector of the selected movie
            n_neighbors=n_neighbors) 

        neighbor_movie_ids = [idx_to_movie[i] for i in indices.flatten()] #map neighbor indices back to movieIds
        neighbor_distances = distances.flatten().tolist()

        neighbor_similarity = [] #use cosine similarity instead of distance for better interpretability
        for dist in neighbor_distances:
            similarity = 1 - dist
            neighbor_similarity.append(similarity)

        recs = []
        for rec_movie_id, similarity in zip(neighbor_movie_ids, neighbor_similarity):
            if rec_movie_id == movie_id: #skips the movie itself in the recommendations
                continue
            recs.append({
                'movieId': rec_movie_id,
                'Title': lookup_df.loc[rec_movie_id, 'title'],
                'Similarity': similarity,
            })
            if len(recs) >= n_recs:
                break
        
        #print(f"Recommendations for '{movie_name}' at movieId {movie_id}:")
        return pd.DataFrame(recs)
    elif model_type.lower() == 'svd':
        #print("Using SVD model for recommendations.")
        movie_vector = matrix[movie_idx].reshape(1, -1)  # Get the latent factors for the selected movie
        similarities = cosine_similarity(movie_vector, matrix).flatten()  # Compare against all movie vectors
        similar_indices = similarities.argsort()[::-1][1:n_recs+1]  # Exclude the movie itself
        
        recs = []
        for idx in similar_indices:
            rec_movie_id = idx_to_movie[idx]
            recs.append({
                'movieId': rec_movie_id,
                'Title': lookup_df.loc[rec_movie_id, 'title'],
                'Similarity': similarities[idx],
            })
        #print(f"Recommendations for '{movie_name}' at movieId {movie_id}:")
        return pd.DataFrame(recs)

In [394]:
#KNN vs SVD: 
sample_title1 = 'Batman'
sample_title2 = 'Toy Story'
sample_title3 = 'Titanic'

knn1 = get_collaborative_recommendations(sample_title1, movie_user_sparse, knn, movie_lookup, 'knn', n_recs=5)
print(knn1)
svd1 = get_collaborative_recommendations(sample_title1, movie_factors, svd,  movie_lookup, 'svd', n_recs=5)
print(svd1)
knn2 = get_collaborative_recommendations(sample_title2, movie_user_sparse, knn, movie_lookup, 'knn', n_recs=5)
print(knn2)
svd2 = get_collaborative_recommendations(sample_title2, movie_factors, svd, movie_lookup, 'svd', n_recs=5)
print(svd2)
knn3 = get_collaborative_recommendations(sample_title3, movie_user_sparse, knn, movie_lookup, 'knn', n_recs=5)
print(knn3)
svd3 = get_collaborative_recommendations(sample_title3, movie_factors, svd, movie_lookup, 'svd', n_recs=5)
print(svd3)

   movieId               Title  Similarity
0      153      Batman Forever    0.717942
1      380           True Lies    0.673136
2      590  Dances with Wolves    0.633634
3      457        The Fugitive    0.628796
4      150           Apollo 13    0.627309
   movieId                       Title  Similarity
0      153              Batman Forever    0.936214
1      380                   True Lies    0.915768
2      165  Die Hard: With a Vengeance    0.856574
3      590          Dances with Wolves    0.835206
4      316                    Stargate    0.834334
   movieId               Title  Similarity
0      260           Star Wars    0.541932
1      780    Independence Day    0.538720
2     1270  Back to the Future    0.519962
3     3114         Toy Story 2    0.518515
4      480       Jurassic Park    0.510147
   movieId                                Title  Similarity
0     3114                          Toy Story 2    0.731222
1     1073  Willy Wonka & the Chocolate Factory    0.65188

Qualitative evaluation of recommendation outputs indicated that the SVD-based recommender produced more coherent movie neighborhoods than the KNN baseline. 
For example, SVD associated Toy Story with Toy Story 2 and A Bug's Life, whereas KNN primarily surfaced broadly popular films. 
Therefore, the SVD recommender was selected as the final model.

In [395]:
# Hybrid Recommendation System

def get_hybrid_recommendations(movie_name, movie_user_sparse,model, keyword_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.5, content_weight=0.5):
    """Return a dataframe of top-N similar movies for a given movie title using both collaborative and content-based filtering."""

    # Detect collaborative model type
    if hasattr(model, 'kneighbors'):
        model_type = 'knn'
    elif hasattr(model, 'components_'):
        model_type = 'svd'
    else:
        raise ValueError("Unsupported collaborative model.")

    # Get recommendations from both systems
    collab_recs = get_collaborative_recommendations( movie_name, movie_user_sparse, model,  movie_lookup,  model_type, n_recs=500)
    content_recs = get_content_based_recommendations(  movie_name,  keyword_matrix, content_lookup, n_recs=500)



    collab_recs = collab_recs.merge(movie_id_map, on='movieId', how='left')

    # Rename columns for clarity
    collab_recs = collab_recs.rename(columns={'Similarity': 'Similarity_collab','Title': 'Title_collab'})
    content_recs = content_recs.rename(columns={'Similarity': 'Similarity_content','Title': 'Title_content' })

    # Keep ALL recommendations from either system
    hybrid = pd.merge(collab_recs, content_recs, on='tmdbId', how='outer')

    # Missing score = movie wasn't recommended by that model
    hybrid['Similarity_collab'] = hybrid['Similarity_collab'].fillna(0)
    hybrid['Similarity_content'] = hybrid['Similarity_content'].fillna(0)

    # Use whichever title exists
    hybrid['Title'] = hybrid['Title_collab'].fillna(hybrid['Title_content'])

    # Normalize scores to 0-1
    scaler = MinMaxScaler()
    hybrid['Similarity_collab'] = scaler.fit_transform( hybrid[['Similarity_collab']] )
    hybrid['Similarity_content'] = scaler.fit_transform(hybrid[['Similarity_content']])

    # Weighted hybrid score
    hybrid['Hybrid_Score'] = (collab_weight * hybrid['Similarity_collab'] + content_weight * hybrid['Similarity_content'])

    # Remove duplicate movies
    hybrid = hybrid.drop_duplicates(subset=['tmdbId'])
    # Sort by hybrid score
    hybrid = hybrid.sort_values(by='Hybrid_Score', ascending=False)

    return hybrid[['tmdbId', 'Title', 'Hybrid_Score']].head(n_recs)

In [396]:
sample_title1 = 'Batman'
sample_title2 = 'Toy Story'
sample_title3 = 'Titanic'

hybrid11 = get_hybrid_recommendations(sample_title1, movie_user_sparse, svd, content_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.7, content_weight=0.3)
print("hybrid11:", hybrid11)
hybrid12 = get_hybrid_recommendations(sample_title1, movie_user_sparse, svd, content_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.3, content_weight=0.7)
print("hybrid12:", hybrid12)

hybrid21 = get_hybrid_recommendations(sample_title2, movie_user_sparse, svd, content_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.7, content_weight=0.3)
print("hybrid21:", hybrid21)
hybrid22 = get_hybrid_recommendations(sample_title2, movie_user_sparse, svd, content_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.3, content_weight=0.7)
print("hybrid22:", hybrid22)

hybrid31 = get_hybrid_recommendations(sample_title3, movie_user_sparse, svd, content_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.7, content_weight=0.3)
print("hybrid31:", hybrid31)
hybrid32 = get_hybrid_recommendations(sample_title3, movie_user_sparse, svd, content_matrix, movie_lookup, content_lookup, n_recs=10, collab_weight=0.3, content_weight=0.7)
print("hybrid32:", hybrid32)

hybrid11:      tmdbId                       Title  Hybrid_Score
105     414              Batman Forever      0.789429
713   36955                   True Lies      0.656314
139     581          Dances with Wolves      0.617799
369    5503                The Fugitive      0.613082
135     568                   Apollo 13      0.611633
263    1572  Die Hard: With a Vengeance      0.606167
351    3049  Ace Ventura: Pet Detective      0.588665
206     812                     Aladdin      0.584038
106     415              Batman & Robin      0.581116
92      329               Jurassic Park      0.579894
hybrid12:      tmdbId                                  Title  Hybrid_Score
106     415                         Batman & Robin      0.820478
97      364                         Batman Returns      0.565126
105     414                         Batman Forever      0.508668
219     854                               The Mask      0.338434
294    1924                               Superman      0.319